In [59]:
using MAT               # pour charger .mat
using LinearAlgebra
using Distributions
using Random
using Symbolics

In [72]:
using ModelingToolkit, Symbolics

# États et paramètres
@variables θ1 θ2 ω1 ω2 Tm1 Tm2 N
@variables PG1 PG2 θ1 θ2 PL1 PL2
@variables KL D1 D2 J1 J2 Ks ω0 a1 a2 b1 b2 Pr1 Pr2 P0

x = [θ1, θ2, ω1, ω2, Tm1, Tm2, N]  # États
u = [PG1, PG2, θ1, θ2, PL1, PL2]                       # Entrées

# Sorties
F12 = KL*(θ1 - θ2)
F21 = KL*(θ2 - θ1)
PL1 = F12 - PG1
PL2 = F21 - PG2
Y = [F12, F21, PG1, PG2, PL1, PL2]

# Système dynamique
ωr = (J1*ω1 + J2*ω2)/(J1+J2)

f = [
    ω1 - ω0,
    ω2 - ω0,
    (Tm1 - (PL1 + F12) - D1*(ω1 - ω0))/J1,
    (Tm2 - (PL2 + F21) - D2*(ω2 - ω0))/J2,
    -a1*(Tm1 - (N*Pr1+P0)*ω1) - b1*(ω1 - ω0),
    -a2*(Tm2 - (N*Pr2+P0)*ω2) - b2*(ω2 - ω0),
    Ks*(ωr - ω0)
]

# Fonction pour calculer la dérivée de Lie
function lie_derivative(h::Vector{Num}, f::Vector{Num}, x::Vector{Num})
    dhdx = [Symbolics.expand_derivatives.(Symbolics.jacobian(h, x))]  # Jacobien
    Lfh = dhdx[1] * f  # dérivée de Lie
    return Lfh
end

# Calcul de la matrice d'observabilité de Lie
function lie_observability(f, Y, x, n_deriv)
    O = Symbolics.zeros(0, length(x))  # matrice vide initiale
    Lh = Y
    for k in 0:n_deriv-1
        # Jacobien pour chaque h
        for h in Lh
            J = Symbolics.jacobian([h], x)
            O = vcat(O, J)
        end
        # Calcul de la prochaine dérivée de Lie
        Lh = lie_derivative(Lh, f, x)
    end
    return O
end

# Calcul avec n dérivées (ici = taille de x = 7)
O_Lie = lie_observability(f, Y, x, length(x))
println("Matrice d'observabilité de Lie :")
# display(O_Lie)

Matrice d'observabilité de Lie :


In [ ]:
using Symbolics, LinearAlgebra

function evaluate_matrix(O_sym::Matrix{Num}, params::Dict{Num, Float64})
    O_num = zeros(Float64, size(O_sym)...)
    for i in 1:size(O_sym,1)
        for j in 1:size(O_sym,2)
            # substitution + extraction de la valeur numérique
            O_num[i,j] = Symbolics.substitute(O_sym[i,j], params) |> Symbolics.value
        end
    end
    return O_num
end

# Exemple
params_num = Dict(
    θ1=>0.0, θ2=>asin((PL0[1] - P0[1]) / KL), ω1=>2π * 49, ω2=>2π * 49, Tm1=>0.5, Tm2=>0.5, N=>1.0,
    PG1=>400.0, PG2=>600.0,
    KL=>3064.0, D1=>0.04, D2=>0.02, J1=>0.4, J2=>0.2, Ks=>0.05, ω0=>2π * 50,
    a1=>100.0, a2=>100.0, b1=>2000.0, b2=>2000.0, Pr1=>100.0, Pr2=>50.0, P0=>1000.0
)

O_num = evaluate_matrix(O_Lie, params_num)
rank_O = rank(O_num)

println("Rang numérique de la matrice d'observabilité : ", rank_O)
println("Dimension du vecteur d'états : ", length(x))

if rank_O == length(x)
    println("✅ Le système est localement observable pour ce point d'équilibre.")
else
    println("❌ Le système n'est pas observable (rang < dimension de l'état).")
end


Rang numérique de la matrice d'observabilité : 4
Dimension du vecteur d'états : 7
❌ Le système n'est pas observable (rang < dimension de l'état).


In [62]:
# @variables θ1 θ2 ω1 ω2 Tm1 Tm2 N           # États
# @variables PG1 PG2                          # Entrées connues
# @variables KL D1 D2 J1 J2 Ks ω0 a1 a2 b1 b2 Pr1 Pr2 P0      # Paramètres
# x = [θ1, θ2, ω1, ω2, Tm1, Tm2, N]
# print("matrice X: ", x)
# u = [PG1, PG2]

# # Sorties mesurées
# F12 = KL*(θ1 - θ2)
# F21 = KL*(θ2 - θ1)
# PL1 = F12 - PG1
# PL2 = F21 - PG2
# print("\n matrice Y:")
# Y = [F12, F21, PG1, PG2, PL1, PL2]


In [63]:
# ωr = (J1*ω1 + J2*ω2)/(J1+J2)

# f = [
#     ω1 - ω0,                                           # θ1_dot
#     ω2 - ω0,                                           # θ2_dot
#     (Tm1 - (PL1 + F12) - D1*(ω1 - ω0))/J1,            # ω1_dot
#     (Tm2 - (PL2 + F21) - D2*(ω2 - ω0))/J2,            # ω2_dot
#     -a1*(Tm1 - (N*Pr1+P0)*ω1) - b1*(ω1 - ω0), # Tm1_dot
#     -a2*(Tm2 - (N*Pr2+P0)*ω2) - b2*(ω2 - ω0), # Tm2_dot
#     Ks*((J1*ω1 + J2*ω2)/(J1+J2) - ω0)                                       # N_dot
# ]

In [64]:
# using Symbolics

# function lie_derivatives(Y, x, f, n)
#     # Liste des dérivées de Lie
#     Lie = [Y]
    
#     println("Ordre 0 : ", Y)  # dérivée initiale
    
#     for k in 1:n
#         # Calculer la jacobienne de la dernière dérivée
#         J = Symbolics.jacobian(Lie[end], x)
        
#         # Vérification dimensionnelle
#         if size(J, 2) != length(f)
#             error("Dimension mismatch: jacobian columns = $(size(J,2)), f length = $(length(f))")
#         end
        
#         # Calcul de la prochaine dérivée de Lie
#         L_next = simplify.(J * f)
#         push!(Lie, L_next)
        
#         # Affichage résumé
#         println("Ordre $k : taille = ", size(L_next))
#         println("jacobian", L_next)
#     end
    
#     println("Nombre total de dérivées calculées : ", length(Lie))
    
#     return Lie
# end


In [65]:
# params = Dict(KL => 3064, D1 =>0.04, D2 =>0.02, J1 =>0.4, J2 =>0.6, Ks =>0.05, ω0 =>314.1592, a1 =>100, a2 =>100, b1 =>2000, b2 =>2000, Pr1 =>100, Pr2 =>50, P0=>400)
# f_simpl = substitute(f, params)
# Y_simpl = substitute(Y, params)

In [66]:
# n_states = length(x)
# Lie_list = lie_derivatives(Y, x, f, n_states-1)
# println(length(Lie_list))
# for (i, L) in enumerate(Lie_list)
#     println(i, ": ", L)
# end


In [67]:


# # Matrice d'observabilité = empilement vertical des jacobiennes
# O = vcat([Symbolics.jacobian(L, x) for L in Lie_list]...)


In [68]:
# # Pour obtenir le rang, on peut substituer des valeurs numériques
# # exemple : θ1=0, θ2=0, ω1=ω0, ω2=ω0, etc.
# subs_dict = Dict(θ1=>0, θ2=>0, ω1=>ω0, ω2=>ω0, Tm1=>1, Tm2=>1, N=>0, PL1=>1, PL2=>1, KL=>1, D1=>0.1, D2=>0.1, J1=>1, J2=>1, Ks=>0.1)
# O_num = substitute.(O, subs_dict)
# rank_O = rank(Matrix(O_num))
# println("Rang de la matrice d'observabilité : ", rank_O)